In [2]:
import duckdb
import pandas as pd
import json
from pathlib import Path


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Change this to your actual Parquet path.
PARQUET_PATH = Path("Data/Global/global_ismn_daily.parquet")

assert PARQUET_PATH.exists(), f"File not found: {PARQUET_PATH}"

print(f"Parquet: {PARQUET_PATH}")
print(f"Size:    {PARQUET_PATH.stat().st_size / 1024**3:.2f} GB")

Parquet: Data/Global/global_ismn_daily.parquet
Size:    0.06 GB


In [4]:
con = duckdb.connect()

# Register the parquet as a view.
# DuckDB reads the Parquet lazily; it does NOT load the whole file into RAM.
con.execute(f"""
    CREATE OR REPLACE VIEW ismn AS
    SELECT *
    FROM read_parquet('{PARQUET_PATH.as_posix()}')
""")

print("Parquet registered successfully.")

Parquet registered successfully.


In [6]:
EXPECTED_COLUMNS = [
    "station_id",
    "network",
    "network_name",
    "station",
    "latitude",
    "longitude",
    "elevation_m",
    "sensor",
    "depth_from_m",
    "depth_to_m",
    "depth_class",
    "sample_date",
    "soil_moisture_m3_m3",
    "observation_count",
    "source_file",
    "clay_pct",
    "sand_pct",
    "silt_pct",
    "climate_zone",
]

schema = con.execute("DESCRIBE ismn").fetchdf()
actual_columns = set(schema["column_name"])

missing = sorted(set(EXPECTED_COLUMNS) - actual_columns)
extra = sorted(actual_columns - set(EXPECTED_COLUMNS))

print("Missing expected columns:", missing)
print("Extra columns:", extra)

Missing expected columns: []
Extra columns: []


In [7]:
basic = con.execute("""
    SELECT
        COUNT(*) AS raw_rows,
        COUNT(DISTINCT station_id) AS unique_stations,
        COUNT(DISTINCT CAST(sample_date AS DATE)) AS unique_dates,
        COUNT(DISTINCT network) AS unique_networks,
        MIN(CAST(sample_date AS DATE)) AS min_date,
        MAX(CAST(sample_date AS DATE)) AS max_date
    FROM ismn
""").df()

display(basic)

,raw_rows,unique_stations,unique_dates,unique_networks,min_date,max_date
0,5045849,2210,4204,61,2015-04-01,2026-10-03


In [8]:
duplicate_grain = con.execute("""
    SELECT
        COUNT(*) AS station_date_groups,
        SUM(CASE WHEN n_rows > 1 THEN 1 ELSE 0 END) AS groups_with_multiple_rows,
        MAX(n_rows) AS max_rows_per_station_date,
        AVG(n_rows) AS mean_rows_per_station_date
    FROM (
        SELECT
            station_id,
            CAST(sample_date AS DATE) AS sample_date,
            COUNT(*) AS n_rows
        FROM ismn
        GROUP BY station_id, CAST(sample_date AS DATE)
    )
""").df()

display(duplicate_grain)

,station_date_groups,groups_with_multiple_rows,max_rows_per_station_date,mean_rows_per_station_date
0,3514643,932167.0,12,1.435665


In [9]:
print("Depth classes:")
display(
    con.execute("""
        SELECT
            depth_class,
            COUNT(*) AS rows,
            COUNT(DISTINCT station_id) AS stations
        FROM ismn
        GROUP BY depth_class
        ORDER BY rows DESC
    """).df()
)

print("\nSensors:")
display(
    con.execute("""
        SELECT
            sensor,
            COUNT(*) AS rows,
            COUNT(DISTINCT station_id) AS stations
        FROM ismn
        GROUP BY sensor
        ORDER BY rows DESC
    """).df()
)

Depth classes:


,depth_class,rows,stations
0,fallback_deeper,2875390,1408
1,near_surface,1902539,1087
2,excluded,267920,329



Sensors:


,sensor,rows,stations
0,Stevens-Hydraprobe-Digital,1844205,498
1,Stevens-Hydraprobe-Analog,1223923,520
2,Meter-5TM,285864,251
3,Acclima-TDT,253043,49
4,Campbell-CS655,156368,67
5,Hydroinnova-CRS-1000B,150523,130
6,Campbell-CS616,120625,49
7,DeltaT-ThetaProbe-ML3,116626,35
8,GPS,108087,152
9,Meter-5TE,98587,50


In [10]:
station_geometry = con.execute("""
    SELECT
        station_id,
        COUNT(DISTINCT ROUND(latitude, 5)) AS unique_latitudes,
        COUNT(DISTINCT ROUND(longitude, 5)) AS unique_longitudes,
        MIN(latitude) AS min_lat,
        MAX(latitude) AS max_lat,
        MIN(longitude) AS min_lon,
        MAX(longitude) AS max_lon
    FROM ismn
    GROUP BY station_id
""").df()

print(
    "Stations with >1 latitude:",
    (station_geometry["unique_latitudes"] > 1).sum()
)

print(
    "Stations with >1 longitude:",
    (station_geometry["unique_longitudes"] > 1).sum()
)

display(
    station_geometry[
        (station_geometry["unique_latitudes"] > 1) |
        (station_geometry["unique_longitudes"] > 1)
    ].head(20)
)

Stations with >1 latitude: 0
Stations with >1 longitude: 0


,station_id,unique_latitudes,unique_longitudes,min_lat,max_lat,min_lon,max_lon


In [11]:
spatial_extent = con.execute("""
    SELECT
        MIN(latitude)  AS min_lat,
        MAX(latitude)  AS max_lat,
        MIN(longitude) AS min_lon,
        MAX(longitude) AS max_lon
    FROM ismn
""").df()

display(spatial_extent)

,min_lat,max_lat,min_lon,max_lon
0,-45.26482,71.3298,-165.12042,175.91821


In [12]:
spatial_grid = con.execute("""
    SELECT
        FLOOR(latitude / 5.0) * 5.0 AS lat_bin_5deg,
        FLOOR(longitude / 5.0) * 5.0 AS lon_bin_5deg,
        COUNT(*) AS raw_rows,
        COUNT(DISTINCT station_id) AS stations,
        COUNT(DISTINCT CONCAT(
            station_id, '|', CAST(CAST(sample_date AS DATE) AS VARCHAR)
        )) AS station_date_samples
    FROM ismn
    GROUP BY
        FLOOR(latitude / 5.0),
        FLOOR(longitude / 5.0)
    ORDER BY station_date_samples DESC
""").df()

display(spatial_grid.head(30))

,lat_bin_5deg,lon_bin_5deg,raw_rows,stations,station_date_samples
0,35.0,-115.0,280160,100,272545
1,40.0,-115.0,278806,128,270911
2,35.0,-110.0,211344,104,196373
3,40.0,-120.0,178328,74,165970
4,35.0,-120.0,173758,72,162486
5,30.0,-100.0,134936,48,129199
6,35.0,-125.0,167546,135,129074
7,40.0,0.0,151442,41,100261
8,30.0,-90.0,121948,42,98861
9,35.0,-100.0,420156,48,95514


In [13]:
year_stats = con.execute("""
    SELECT
        YEAR(CAST(sample_date AS DATE)) AS year,
        COUNT(*) AS raw_rows,
        COUNT(DISTINCT station_id) AS stations,
        COUNT(DISTINCT CONCAT(
            station_id, '|', CAST(CAST(sample_date AS DATE) AS VARCHAR)
        )) AS station_date_samples
    FROM ismn
    GROUP BY YEAR(CAST(sample_date AS DATE))
    ORDER BY year
""").df()

display(year_stats)

,year,raw_rows,stations,station_date_samples
0,2015,448410,1575,329903
1,2016,560385,1583,394790
2,2017,538455,1609,369953
3,2018,520949,1422,344806
4,2019,485878,1383,308679
5,2020,417573,1092,269853
6,2021,394968,1057,259625
7,2022,395350,1113,260490
8,2023,326909,1034,233307
9,2024,353628,1033,264946


In [14]:
climate_stats = con.execute("""
    SELECT
        climate_zone,
        COUNT(*) AS raw_rows,
        COUNT(DISTINCT station_id) AS stations,
        COUNT(DISTINCT CONCAT(
            station_id, '|', CAST(CAST(sample_date AS DATE) AS VARCHAR)
        )) AS station_date_samples
    FROM ismn
    GROUP BY climate_zone
    ORDER BY station_date_samples DESC
""").df()

display(climate_stats)

,climate_zone,raw_rows,stations,station_date_samples
0,BSk,785753,327,670868
1,Dfb,1019524,473,659586
2,Cfa,988743,225,555028
3,Cfb,520808,141,215879
4,Dsb,231396,106,215438
5,Dfa,231923,78,188938
6,Dfc,208595,121,170779
7,Csa,183876,146,140470
8,Csb,117713,60,99207
9,BWk,107029,53,95449


In [15]:
con.execute("""
    CREATE OR REPLACE TABLE station_date_candidates AS

    SELECT
        station_id,

        ANY_VALUE(network) AS network,
        ANY_VALUE(network_name) AS network_name,
        ANY_VALUE(station) AS station,

        AVG(latitude) AS latitude,
        AVG(longitude) AS longitude,
        AVG(elevation_m) AS elevation_m,

        ANY_VALUE(climate_zone) AS climate_zone,

        CAST(sample_date AS DATE) AS sample_date,

        COUNT(*) AS source_rows,

        MIN(depth_from_m) AS min_depth_from_m,
        MAX(depth_to_m) AS max_depth_to_m,

        MAX(observation_count) AS max_observation_count,

        STRING_AGG(
            DISTINCT CAST(sensor AS VARCHAR),
            ', '
        ) AS sensors,

        STRING_AGG(
            DISTINCT CAST(depth_class AS VARCHAR),
            ', '
        ) AS depth_classes

    FROM ismn

    GROUP BY
        station_id,
        CAST(sample_date AS DATE)
""")

candidate_count = con.execute("""
    SELECT COUNT(*) FROM station_date_candidates
""").fetchone()[0] # type: ignore

print(f"Unique station-date candidates: {candidate_count:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique station-date candidates: 3,514,643


In [16]:
display(
    con.execute("""
        SELECT *
        FROM station_date_candidates
        ORDER BY sample_date
        LIMIT 20
    """).df()
)

,station_id,network,network_name,station,latitude,longitude,elevation_m,climate_zone,sample_date,source_rows,min_depth_from_m,max_depth_to_m,max_observation_count,sensors,depth_classes
0,HOBE__1.07,HOBE,HOBE,1.07,56.04260,9.14130,60.00,Dfb,2015-04-01,2,0.0000,0.0500,24,Meter-5TE,near_surface
1,SNOTEL__Butte,SNOTEL,SNOTEL,Butte,38.89435,-106.95327,3106.00,Dfc,2015-04-01,1,0.0508,0.0508,9,Stevens-Hydraprobe-Analog,fallback_deeper
2,SNOTEL__Galena_Summit,SNOTEL,SNOTEL,Galena_Summit,43.87497,-114.71363,2676.00,Dsb,2015-04-01,1,0.0508,0.0508,2,Stevens-Hydraprobe-Analog,fallback_deeper
3,SOILSCAPE__node502,SOILSCAPE,SOILSCAPE,node502,38.14887,-120.78742,209.00,Csa,2015-04-01,1,0.0500,0.0500,24,Meter-EC5,near_surface
4,CTP_SMTMN__L29,CTP_SMTMN,CTP_SMTMN,L29,31.58700,92.24100,4637.00,ET,2015-04-01,2,0.0000,0.1000,7,Meter-5TM,"near_surface, fallback_deeper"
5,RSMN__Satu_Mare,RSMN,RSMN,Satu_Mare,47.72144,22.88737,123.00,Dfb,2015-04-01,1,0.0000,0.0500,23,Meter-5TM,near_surface
6,COSMOS-UK__The_Lizard,COSMOS-UK,COSMOS-UK,The_Lizard,50.03281,-5.19988,85.00,masked,2015-04-01,2,0.1000,0.1000,24,Acclima-TDT,fallback_deeper
7,PBO_H2O__FREEMONTAP,PBO_H2O,PBO_H2O,FREEMONTAP,38.42175,-105.10468,1616.90,BSk,2015-04-01,1,0.0000,0.0500,1,GPS,near_surface
8,SCAN__Lovelock_NNR,SCAN,SCAN,Lovelock_NNR,40.09247,-118.49345,1199.00,BWk,2015-04-01,1,0.0508,0.0508,21,Stevens-Hydraprobe-Digital,fallback_deeper
9,SNOTEL__GBRC_HQ,SNOTEL,SNOTEL,GBRC_HQ,39.32019,-111.48827,2682.00,Dfb,2015-04-01,1,0.0508,0.0508,7,Stevens-Hydraprobe-Analog,fallback_deeper


In [17]:
TARGET_BENCHMARK_SIZE = 10_000
SEED = 42

n_years = con.execute("""
    SELECT COUNT(DISTINCT YEAR(sample_date))
    FROM station_date_candidates
""").fetchone()[0] # type: ignore

per_year = max(1, (TARGET_BENCHMARK_SIZE + n_years - 1) // n_years)

print("Number of years:", n_years)
print("Approx samples/year:", per_year)

Number of years: 12
Approx samples/year: 834


In [18]:
con.execute(f"""
    CREATE OR REPLACE TABLE benchmark_10000 AS

    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY YEAR(sample_date)
                ORDER BY
                    HASH(
                        CONCAT(
                            CAST(station_id AS VARCHAR),
                            '|',
                            CAST(sample_date AS VARCHAR),
                            '|',
                            '{SEED}'
                        )
                    )
            ) AS rn
        FROM station_date_candidates
    )
    WHERE rn <= {per_year}
    ORDER BY
        YEAR(sample_date),
        HASH(
            CONCAT(
                CAST(station_id AS VARCHAR),
                '|',
                CAST(sample_date AS VARCHAR),
                '|',
                '{SEED}'
            )
        )
    LIMIT {TARGET_BENCHMARK_SIZE}
""")

benchmark = con.execute("""
    SELECT *
    FROM benchmark_10000
""").df()

print(f"Benchmark samples: {len(benchmark):,}")
display(benchmark.head())

Benchmark samples: 10,000


,station_id,network,network_name,station,latitude,longitude,elevation_m,climate_zone,sample_date,source_rows,min_depth_from_m,max_depth_to_m,max_observation_count,sensors,depth_classes,rn
0,SCAN__Bushland_#1,SCAN,SCAN,Bushland_#1,35.16667,-102.10000,1164.0,BSk,2015-06-14,1,0.0508,0.0508,23,Stevens-Hydraprobe-Analog,fallback_deeper,1
1,TxSON__CR200_6,TxSON,TxSON,CR200_6,30.23830,-98.70370,450.0,Cfa,2015-07-13,1,0.0500,0.0500,24,Campbell-CS655,near_surface,2
2,TWENTE__ITCSM_16b,TWENTE,TWENTE,ITCSM_16b,52.36781,6.53525,8.0,Cfb,2015-10-24,2,0.0500,0.1000,24,Meter-5TM,"fallback_deeper, near_surface",3
3,PBO_H2O__CRRS_SCGN,PBO_H2O,PBO_H2O,CRRS_SCGN,33.06981,-115.73504,-81.9,BWh,2015-12-15,1,0.0000,0.0500,1,GPS,near_surface,4
4,PBO_H2O__KEMMERERLF,PBO_H2O,PBO_H2O,KEMMERERLF,41.74982,-110.51281,2149.8,BSk,2015-04-01,1,0.0000,0.0500,1,GPS,near_surface,5


In [19]:
benchmark_years = con.execute("""
    SELECT
        YEAR(sample_date) AS year,
        COUNT(*) AS samples
    FROM benchmark_10000
    GROUP BY YEAR(sample_date)
    ORDER BY year
""").df()

display(benchmark_years)

,year,samples
0,2015,834
1,2016,834
2,2017,834
3,2018,834
4,2019,834
5,2020,834
6,2021,834
7,2022,834
8,2023,834
9,2024,834


In [20]:
benchmark_climate = con.execute("""
    SELECT
        climate_zone,
        COUNT(*) AS samples
    FROM benchmark_10000
    GROUP BY climate_zone
    ORDER BY samples DESC
""").df()

display(benchmark_climate)

,climate_zone,samples
0,BSk,1892
1,Dfb,1836
2,Cfa,1615
3,Dsb,656
4,Cfb,568
5,Dfa,540
6,Dfc,508
7,Csa,393
8,BWk,280
9,Csb,273


In [21]:
con.execute("""
    CREATE OR REPLACE TABLE benchmark_coverage AS

    SELECT *
    FROM (
        SELECT
            *,
            FLOOR(latitude / 10.0) * 10.0 AS lat_bin_10deg,
            FLOOR(longitude / 10.0) * 10.0 AS lon_bin_10deg,

            ROW_NUMBER() OVER (
                PARTITION BY
                    YEAR(sample_date),
                    climate_zone,
                    FLOOR(latitude / 10.0),
                    FLOOR(longitude / 10.0)

                ORDER BY
                    HASH(
                        CONCAT(
                            CAST(station_id AS VARCHAR),
                            '|',
                            CAST(sample_date AS VARCHAR),
                            '|',
                            'coverage_42'
                        )
                    )
            ) AS rn

        FROM station_date_candidates
    )
    WHERE rn = 1
""")

coverage_count = con.execute("""
    SELECT COUNT(*)
    FROM benchmark_coverage
""").fetchone()[0]

print(f"Coverage benchmark size: {coverage_count:,}")

Coverage benchmark size: 1,609


In [22]:
con.execute("""
    CREATE OR REPLACE TABLE benchmark_coverage AS

    SELECT *
    FROM (
        SELECT
            *,
            FLOOR(latitude / 10.0) * 10.0 AS lat_bin_10deg,
            FLOOR(longitude / 10.0) * 10.0 AS lon_bin_10deg,

            ROW_NUMBER() OVER (
                PARTITION BY
                    YEAR(sample_date),
                    climate_zone,
                    FLOOR(latitude / 10.0),
                    FLOOR(longitude / 10.0)

                ORDER BY
                    HASH(
                        CONCAT(
                            CAST(station_id AS VARCHAR),
                            '|',
                            CAST(sample_date AS VARCHAR),
                            '|',
                            'coverage_42'
                        )
                    )
            ) AS rn

        FROM station_date_candidates
    )
    WHERE rn = 1
""")

coverage_count = con.execute("""
    SELECT COUNT(*)
    FROM benchmark_coverage
""").fetchone()[0]

print(f"Coverage benchmark size: {coverage_count:,}")

Coverage benchmark size: 1,609


In [23]:
coverage_stats = con.execute("""
    SELECT
        COUNT(DISTINCT station_id) AS stations,
        COUNT(DISTINCT climate_zone) AS climate_zones,
        COUNT(DISTINCT YEAR(sample_date)) AS years,
        MIN(sample_date) AS min_date,
        MAX(sample_date) AS max_date,
        MIN(latitude) AS min_lat,
        MAX(latitude) AS max_lat,
        MIN(longitude) AS min_lon,
        MAX(longitude) AS max_lon
    FROM benchmark_coverage
""").df()

display(coverage_stats)

,stations,climate_zones,years,min_date,max_date,min_lat,max_lat,min_lon,max_lon
0,696,27,12,2015-04-01,2026-09-29,-45.26482,71.3298,-164.92472,175.87642


In [24]:
benchmark_10000 = con.execute("""
    SELECT *
    FROM benchmark_10000
""").df()

benchmark_coverage = con.execute("""
    SELECT *
    FROM benchmark_coverage
""").df()

benchmark_10000.to_parquet(
    "benchmark_10000.parquet",
    index=False
)

benchmark_coverage.to_parquet(
    "benchmark_coverage.parquet",
    index=False
)

print("Saved:")
print("  benchmark_10000.parquet")
print("  benchmark_coverage.parquet")

Saved:
  benchmark_10000.parquet
  benchmark_coverage.parquet


In [26]:
station_date_stats = con.execute("""
    SELECT
        COUNT(*) AS unique_station_date_samples
    FROM (
        SELECT DISTINCT
            station_id,
            CAST(sample_date AS DATE) AS sample_date
        FROM ismn
    )
""").df()

profile = {
    "file": str(PARQUET_PATH),
    "file_size_gb": PARQUET_PATH.stat().st_size / 1024**3,

    "raw_rows": int(basic.loc[0, "raw_rows"]),
    "unique_stations": int(basic.loc[0, "unique_stations"]),
    "unique_dates": int(basic.loc[0, "unique_dates"]),
    "unique_networks": int(basic.loc[0, "unique_networks"]),

    "min_date": str(basic.loc[0, "min_date"]),
    "max_date": str(basic.loc[0, "max_date"]),

    "unique_station_date_samples": int(
        station_date_stats.loc[0, "unique_station_date_samples"]
    ),

    "groups_with_multiple_rows": int(
        duplicate_grain.loc[0, "groups_with_multiple_rows"]
    ),

    "max_rows_per_station_date": int(
        duplicate_grain.loc[0, "max_rows_per_station_date"]
    ),

    "mean_rows_per_station_date": float(
        duplicate_grain.loc[0, "mean_rows_per_station_date"]
    ),

    "missing_expected_columns": missing,

    "spatial_extent": spatial_extent.iloc[0].to_dict(),

    "n_years": int(n_years),

    "benchmark_10000_size": int(len(benchmark_10000)),
    "benchmark_coverage_size": int(len(benchmark_coverage)),
}

with open("parquet_profile.json", "w") as f:
    json.dump(profile, f, indent=2, default=str)

print(json.dumps(profile, indent=2, default=str))

{
  "file": "Data/Global/global_ismn_daily.parquet",
  "file_size_gb": 0.05666174925863743,
  "raw_rows": 5045849,
  "unique_stations": 2210,
  "unique_dates": 4204,
  "unique_networks": 61,
  "min_date": "2015-04-01 00:00:00",
  "max_date": "2026-10-03 00:00:00",
  "unique_station_date_samples": 3514643,
  "groups_with_multiple_rows": 932167,
  "max_rows_per_station_date": 12,
  "mean_rows_per_station_date": 1.4356647318091766,
  "missing_expected_columns": [],
  "spatial_extent": {
    "min_lat": -45.26482,
    "max_lat": 71.3298,
    "min_lon": -165.12042,
    "max_lon": 175.91821
  },
  "n_years": 12,
  "benchmark_10000_size": 10000,
  "benchmark_coverage_size": 1609
}
